In [18]:
%%writefile hola.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <climits>
using namespace std;

struct Edge { int to, w; };

int main() {
    const int N = 5;
    const char name[N] = {'a', 'b', 'c', 'd', 'e'};
    vector<Edge> adj[N];
    auto add = [&](int u, int v, int w) {
        adj[u].push_back({v, w});
        adj[v].push_back({u, w});
    };
    // Grafo del Ejercicio 1
    add(0, 1, 5); add(0, 4, 2); add(0, 2, 7);   // a-b 5, a-e 2, a-c 7
    add(1, 4, 3); add(1, 3, 6);                 // b-e 3, b-d 6
    add(2, 4, 4); add(2, 3, 4);                 // c-e 4, c-d 4
    add(3, 4, 5);                               // d-e 5

    vector<int> dist(N, INT_MAX), parent(N, -1);
    vector<bool> inTree(N, false);
    priority_queue<pair<int,int>, vector<pair<int,int>>, greater<>> pq;  // (peso, vertice)

    int start = 0, total = 0;
    dist[start] = 0;
    pq.push({0, start});

    cout << "Prim desde 'a'\n";
    cout << "--------------------------------------------\n";
    while (!pq.empty()) {
        auto [d, u] = pq.top(); pq.pop();
        if (inTree[u]) continue;                // entrada vieja
        inTree[u] = true;
        total += d;
        if (parent[u] >= 0)
            cout << "Agrego " << name[u] << " via arista " << name[parent[u]] << "-" << name[u]
                 << " (peso " << d << ")\n";
        else
            cout << "Raiz: " << name[u] << "\n";
        for (auto &e : adj[u])
            if (!inTree[e.to] && e.w < dist[e.to]) {
                dist[e.to] = e.w; parent[e.to] = u;
                pq.push({e.w, e.to});
            }
        cout << "  Cola (vertice:prioridad):";
        for (int v = 0; v < N; v++)
            if (!inTree[v]) {
                cout << " " << name[v] << ":";
                if (dist[v] == INT_MAX) cout << "inf";
                else cout << dist[v] << "(" << name[parent[v]] << ")";
            }
        cout << "\n";
    }
    cout << "--------------------------------------------\n";
    cout << "Peso total del MST: " << total << "\n";
    return 0;
}

Overwriting hola.cpp


In [19]:
!g++ hola.cpp -o hola
!./hola

Prim desde 'a'
--------------------------------------------
Raiz: a
  Cola (vertice:prioridad): b:5(a) c:7(a) d:inf e:2(a)
Agrego e via arista a-e (peso 2)
  Cola (vertice:prioridad): b:3(e) c:4(e) d:5(e)
Agrego b via arista e-b (peso 3)
  Cola (vertice:prioridad): c:4(e) d:5(e)
Agrego c via arista e-c (peso 4)
  Cola (vertice:prioridad): d:4(c)
Agrego d via arista c-d (peso 4)
  Cola (vertice:prioridad):
--------------------------------------------
Peso total del MST: 13


In [20]:
%%writefile hola.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <string>
#include <climits>
using namespace std;

struct Edge { int to, w; };

struct Graph {
    int n;
    vector<char> name;
    vector<vector<Edge>> adj;
    Graph(string names) : n(names.size()), name(names.begin(), names.end()), adj(names.size()) {}
    int id(char c) const { for (int i = 0; i < n; i++) if (name[i] == c) return i; return -1; }
    void arc(char u, char v, int w)  { adj[id(u)].push_back({id(v), w}); }   // dirigida
    void edge(char u, char v, int w) { arc(u, v, w); arc(v, u, w); }         // no dirigida
};

void dijkstra(const Graph &g, char src) {
    vector<int> dist(g.n, INT_MAX), parent(g.n, -1);
    vector<bool> done(g.n, false);
    priority_queue<pair<int,int>, vector<pair<int,int>>, greater<>> pq;
    int s = g.id(src);
    dist[s] = 0; pq.push({0, s});
    cout << "Orden en que se fijan los vertices: ";
    while (!pq.empty()) {
        auto [d, u] = pq.top(); pq.pop();
        if (done[u]) continue;
        done[u] = true;
        cout << g.name[u] << "(" << d << ") ";
        for (auto &e : g.adj[u])
            if (!done[e.to] && d + e.w < dist[e.to]) {
                dist[e.to] = d + e.w; parent[e.to] = u;
                pq.push({dist[e.to], e.to});
            }
    }
    cout << "\n\nVertice | Distancia | Camino\n";
    for (int v = 0; v < g.n; v++) {
        cout << "   " << g.name[v] << "    |    ";
        if (dist[v] == INT_MAX) { cout << "inf" << "    | (no alcanzable)\n"; continue; }
        cout << dist[v] << "      | ";
        string path;
        for (int x = v; x != -1; x = parent[x])
            path = string(1, g.name[x]) + (path.empty() ? "" : " -> " + path);
        cout << path << "\n";
    }
}

int main() {
    // 3a: grafo dirigido (segun la figura del PDF)
    Graph a("abcde");
    a.arc('b','a',3); a.arc('b','c',4); a.arc('d','b',2); a.arc('d','c',5);
    a.arc('a','d',7); a.arc('c','e',6); a.arc('e','d',4);
    cout << "===== 3a: grafo dirigido, origen a =====\n";
    dijkstra(a, 'a');

    // 3b: grafo no dirigido, 12 vertices
    Graph b("abcdefghijkl");
    b.edge('a','b',3); b.edge('a','c',5); b.edge('a','d',4);
    b.edge('b','e',3); b.edge('b','f',6);
    b.edge('c','d',2); b.edge('d','e',1); b.edge('e','f',2);
    b.edge('c','g',4); b.edge('d','h',5); b.edge('e','i',4); b.edge('f','j',5);
    b.edge('g','h',3); b.edge('h','i',6); b.edge('i','j',3);
    b.edge('g','k',6); b.edge('h','k',7); b.edge('i','l',5); b.edge('j','l',9);
    b.edge('k','l',8);
    cout << "\n===== 3b: grafo no dirigido, origen a =====\n";
    dijkstra(b, 'a');
    return 0;
}

Overwriting hola.cpp


In [21]:
!g++ hola.cpp -o hola
!./hola

===== 3a: grafo dirigido, origen a =====
Orden en que se fijan los vertices: a(0) d(7) b(9) c(12) e(18) 

Vertice | Distancia | Camino
   a    |    0      | a
   b    |    9      | a -> d -> b
   c    |    12      | a -> d -> c
   d    |    7      | a -> d
   e    |    18      | a -> d -> c -> e

===== 3b: grafo no dirigido, origen a =====
Orden en que se fijan los vertices: a(0) b(3) d(4) c(5) e(5) f(7) g(9) h(9) i(9) j(12) l(14) k(15) 

Vertice | Distancia | Camino
   a    |    0      | a
   b    |    3      | a -> b
   c    |    5      | a -> c
   d    |    4      | a -> d
   e    |    5      | a -> d -> e
   f    |    7      | a -> d -> e -> f
   g    |    9      | a -> c -> g
   h    |    9      | a -> d -> h
   i    |    9      | a -> d -> e -> i
   j    |    12      | a -> d -> e -> f -> j
   k    |    15      | a -> c -> g -> k
   l    |    14      | a -> d -> e -> i -> l
